# Extract YouTube Transcripts

- **Tasks:**
    
    1. Find any YouTube clip where predictions exists. For ex, [Immediate 2025 NCAA tournament Final Four and championship picks](https://youtu.be/-rjnvL9LL3U?si=QMFJYAQ8Q85lTNCD). Below, we include all the videos we extract data from.
    2. Use the [`youtube-transcript-api`](https://pypi.org/project/youtube-transcript-api/) or [`whisper`](https://huggingface.co/openai/whisper-large) to retrieve the transcript.
    3. Extract the transcript snippets.
    4. Save the raw transcript snippets.

In [1]:
import os
import sys

import pandas as pd

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import NoTranscriptFound

import yt_dlp
from transformers import pipeline

from typing import Any, Mapping
import math

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Preprocessing code

- Will use the below class and functions for both video transcripts.

---

1. `YouTubeTranscriptApi()`: fetch the YT transcripts.
2. `extract_data()`: transform data into Pandas DataFrame.
3. `fetch_en_auto_transcript_snippets(ytt_api, video_id)`: lists available transcripts for the "video_id", selects **English (auto-generated)** when possible ("language_code == "en" and "is_generated == True"), returns a list of "FetchedTranscriptSnippet" objects (each snippet has "text", "start", "duration").
4. `extract_data(snippets)`: converts the list of "FetchedTranscriptSnippet" objects into a Pandas DataFrame, output columns: "Text", "Start Time", "Duration" (and we add "Video ID" for consistency).
5. `download_audio(video_id, output_path=...)`: downloads the best available audio stream for that YouTube video (using "yt-dlp"), saves it to "output_path" (e.g. ".../{video_id}.webm")
6. `whisper_result_to_dataframe(result, video_id)`: converts Whisper output (with timestamps) into a Pandas DataFrame, output columns: "Text", "Start Time", "Duration", "Video ID".

In [3]:
ytt_api = YouTubeTranscriptApi()

In [4]:
def extract_data(snippets) -> pd.DataFrame:
    """Extract the text, start time, and duration from the YT `FetchedTranscriptSnippet()`."""
    snippet_texts = []
    snippet_starts = []
    snippet_durations = []
    data = {}

    for snippets_idx in range(len(snippets)):
        snippet = snippets[snippets_idx]
        snippet_texts.append(snippet.text)
        snippet_starts.append(snippet.start)
        snippet_durations.append(snippet.duration)

    data['Text'] = snippet_texts
    data['Start Time'] = snippet_starts
    data['Duration'] = snippet_durations

    df = pd.DataFrame(data=data)
    return df

In [5]:
def fetch_en_auto_transcript_snippets(ytt_api, video_id: str):
    
    tl = ytt_api.list(video_id)
    
    for t in tl:
        if t.language == "English (auto-generated)" and t.language_code == "en" and t.is_generated:
            return t.fetch()
            
    for t in tl:
        if t.language_code == "en" and t.is_generated:
            return t.fetch()
    
    raise NoTranscriptFound(video_id)

In [6]:
def download_audio(video_id, output_path="audio.%(ext)s"):
    """
    Pass a YouTube VideoID and download the audio file.
    """
    url = f"https://www.youtube.com/watch?v={video_id}"
    
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': output_path,
        'quiet': True
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

In [7]:
def whisper_result_to_dataframe(result: Mapping[str, Any], video_id: str) -> pd.DataFrame:
    """
    Converts the dict from pipeline Whisper into a DataFrame:
    Text, Start Time, Duration, Video ID.
    """
    rows = []
    chunks = result.get("chunks") or []
    for chunk in chunks:
        text = (chunk.get("text") or "").strip()
        ts = chunk.get("timestamp")
        if not text or ts is None:
            continue
        start_s = end_s = None
        if isinstance(ts, (list, tuple)):
            if len(ts) >= 2:
                a, b = ts[0], ts[1]
                
                if a is not None and b is not None:
                    try:
                        start_s = float(a)
                        end_s = float(b)
                    except (TypeError, ValueError):
                        continue
                
                elif a is not None and b is None:
                    continue  
            elif len(ts) == 1 and ts[0] is not None:
                try:
                    start_s = float(ts[0])
                    end_s = start_s  
                except (TypeError, ValueError):
                    continue
        elif isinstance(ts, dict):
            a = ts.get("start")
            b = ts.get("end")
            if a is None or b is None:
                continue
            try:
                start_s = float(a)
                end_s = float(b)
            except (TypeError, ValueError):
                continue
        else:
            continue
        if start_s is None or end_s is None:
            continue
        if not math.isfinite(start_s) or not math.isfinite(end_s):
            continue
        duration = end_s - start_s
        if duration < 0:
            continue
        rows.append({
            "Text": text,
            "Start Time": start_s,
            "Duration": duration,
            "Video ID": video_id,
        })
    return pd.DataFrame(rows)

In [8]:
def extract_transcript_df(
    video_id: str,
    *,
    extract_with_whisper: bool = False,
    ytt_api: YouTubeTranscriptApi | None = None,
    whisper_pipe=None,
    audio_dir: str | None = None,
    audio_ext: str = "webm",
    whisper_model: str = "openai/whisper-large-v3",
) -> pd.DataFrame:
    """
    Unified transcript extractor.
    - If extract_with_whisper=False:
        uses YouTubeTranscriptApi + your English(auto-generated) filter
    - If extract_with_whisper=True:
        downloads audio and transcribes with Whisper pipeline
    """
    if not extract_with_whisper:
        if ytt_api is None:
            ytt_api = YouTubeTranscriptApi()
        
        snippets = fetch_en_auto_transcript_snippets(ytt_api, video_id)
        
        df = extract_data(snippets)
        
        if "Video ID" not in df.columns:
            df["Video ID"] = video_id
        return df[["Text", "Start Time", "Duration", "Video ID"]]
    
    if audio_dir is None:
        audio_dir = os.getcwd()
    os.makedirs(audio_dir, exist_ok=True)
    audio_path = os.path.join(audio_dir, f"{video_id}.{audio_ext}")
    
    download_audio(video_id, output_path=audio_path)
    
    if whisper_pipe is None:
        whisper_pipe = pipeline(
            "automatic-speech-recognition",
            model=whisper_model,
            return_timestamps=True,
        )
    result = whisper_pipe(audio_path)
    
    return whisper_result_to_dataframe(result, video_id)

## Transcripts 

| # | Title | Link | Video ID | DOMAIN |
|---|---|---|---|---|
| 1 | Immediate 2025 NCAA tournament Final Four and championship picks | https://www.youtube.com/watch?v=-rjnvL9LL3U | -rjnvL9LL3U | SPORTS |
| 2 | FULL PREVIEW & PICKS: Patriots vs. Seahawks Super Bowl LX 🏆 Who wins the Lombardi Trophy? \| NFL Live | https://www.youtube.com/watch?v=ZZN7BAYeOtc | ZZN7BAYeOtc | SPORTS |
| 3 | FIRST TAKE'S SUPER BOWL PICKS! The crew is going with... 😱 | https://www.youtube.com/watch?v=mBK8o5orBbE | mBK8o5orBbE | SPORTS |
| 4 | NFL Predictions and Picks For Super Bowl LX [Patriots vs Seahawks] - Best Bets ✅ | https://www.youtube.com/watch?v=LXPQrZV4Cfw | LXPQrZV4Cfw | SPORTS |
| 5 | Rich Eisen’s Pick to Win the Seahawks vs Patriots Super Bowl LX Is….? - The Rich Eisen Show | https://www.youtube.com/watch?v=fUmJAtFEGn8 | fUmJAtFEGn8 | SPORTS |
| 6 | The Pat McAfee Show's Picks For Super Bowl LX | https://www.youtube.com/watch?v=MTVAkVkkaz4 | MTVAkVkkaz4 | SPORTS |
| 7 | Super Bowl LX On-Site Preview: Picks, Predictions, Everything you need to know for Patriots-Seahawks | https://www.youtube.com/watch?v=Z0xP3GNpjkw | Z0xP3GNpjkw | SPORTS |
| 8 | Sharp 600 - BEST SBLX Patriots vs. Seahawks Any Time Touchdown Scorers & Props | https://www.youtube.com/watch?v=6STv2GFNB6I | 6STv2GFNB6I | SPORTS |
| 9 | Super Bowl LX - Patriots vs. Seahawks - Picks and Predictions w/ Todd Fuhrman - Sharp 600 | https://www.youtube.com/watch?v=FPl-F2k_KtM | FPl-F2k_KtM | SPORTS |
| 10 | Predicting the UEFA Champions League TO THE FINAL | https://www.youtube.com/watch?v=AoE8KFAXHSc | AoE8KFAXHSc | SPORTS |
| 11 | Insane Tennis Predictions For 2026 | https://www.youtube.com/watch?v=jCe-bY1nP7o | jCe-bY1nP7o | SPORTS |
| 12 | Morgan Stanley's Wilson Bullish on Stocks for 2026 | https://www.youtube.com/watch?v=sj8tHn6lGAs | sj8tHn6lGAs | FINANCE |
| 13 | Our 2026 Financial Predictions | https://www.youtube.com/watch?v=mkNOcK-S8XQ | mkNOcK-S8XQ | FINANCE |
| 14 | Spring Forecast 2026: Wintry Weather Isn’t Finished Yet! | https://www.youtube.com/watch?v=ysSuV0_vnYI | ysSuV0_vnYI | WEATHER |
| 15 | 2026 Weather Outlook: La Niña’s Exit, El Niño’s Potential and the Signals Farmers Should Watch | https://www.youtube.com/watch?v=nGTBk-VI4Ew | nGTBk-VI4Ew | WEATHER |
| 16 | Farmers' Almanac Winter Weather Forecast 2025 - 2026 | https://www.youtube.com/watch?v=nsdAJlyjVeA | nsdAJlyjVeA | WEATHER |
| 17 | Economist makes dire prediction about US employment rate | https://www.youtube.com/watch?v=DieTKcXFyi8 | DieTKcXFyi8 | POLITICAL |
| 18 | JP Morgan strategist predicts 2026 inflation outlook | https://www.youtube.com/watch?v=l4Gdl6SCTQg | l4Gdl6SCTQg | POLITICAL |
| 19 | JPMorgan releases new prediction for the US economy in 2026 | https://www.youtube.com/watch?v=hdib59Tj76E | hdib59Tj76E | POLITICAL |
| 20 | This Fed will remain ‘paralyzed’: Expert makes prediction on future rate hikes | https://www.youtube.com/watch?v=m8tsZKtdtME | m8tsZKtdtME | POLITICAL |
| 21 | 2026 HOUSE Prediction Map Based on NEW POLLS! | https://www.youtube.com/watch?v=DEhAFSPp6lQ | DEhAFSPp6lQ | POLITICAL |
| 22 | Saagar REVEALS 2024 PREDICTION: Trump WINS | https://www.youtube.com/watch?v=KIwCGebr5xg | KIwCGebr5xg | POLITICAL |
| 23 | American Cancer Society Predicting More Cancer Cases in 2026 | https://www.youtube.com/watch?v=oSIQ8l6SqAc | oSIQ8l6SqAc | HEALTH |
| 24 | Doctors predict uterine cancer rates to rise | https://www.youtube.com/watch?v=WIxqbAJw5hU | WIxqbAJw5hU | HEALTH |
| 25 | Just over a third of U.S. adults are obese. By 2030, 42 percent will be, says a forecast released Mo | https://www.youtube.com/watch?v=Y8epuW7-TXw | Y8epuW7-TXw | HEALTH |

## Extracting Transcriptions

There are **two ways** to obtain a transcript for a YouTube video:

- **YouTube transcripts (fast, no download)**: uses `YouTubeTranscriptApi()` to fetch captions that already exist on YouTube. In this notebook, we prioritize **English auto-generated** captions when available.

- **Whisper transcripts (slower, download required)**: downloads the video audio and transcribes it locally using a Whisper model via the Hugging Face `pipeline(...)`.

To make this easy to use, we wrap both approaches in a single function: `extract_transcript_df(...)`.

---

**Option A — Do NOT use Whisper**

Use this when you want the fastest approach, the video has English auto-generated captions availabl and you do not have enough memory available.

```python
df = extract_transcript_df("Z0xP3GNpjkw", extract_with_whisper=False)
df.head()
```

**Option B — Use Whisper (download + transcribe)**

Use this when the video has no captions / captions are disabled, or
you want transcripts that don’t depend on YouTube caption (more reliable)

```python
from transformers import pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    return_timestamps=True
)
df = extract_transcript_df(
    "Z0xP3GNpjkw",
    extract_with_whisper=True,
    whisper_pipe=pipe,
    audio_dir=save_data_path
)
df.head()
```

---
Computational note about whisper-large-v3: openai/whisper-large-v3 is high quality but computationally heavy.

---
**Saving the final DataFrame**

After you generate the transcript DataFrame (either method), save it so you don’t need to re-run extraction:

```python
base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
save_data_path = os.path.join(base_data_path, "yt", "transcripts")  # choose a folder name you like
DataProcessing.save_to_file(
    df,
    path=save_data_path,
    prefix=video_id,
    save_file_type="csv",
    include_version=False
)
```

This will create a CSV for that "video_id" that you can reuse later in your pipeline.

---

**Whisper setup (only needed if using `extract_with_whisper=True`)**

Whisper transcription in this notebook uses the Hugging Face `pipeline("automatic-speech-recognition", ...)`.
If you plan to use Whisper, make sure you have the required dependencies installed.

Install Python packages:

```python
#!pip install -U transformers accelerate yt-dlp
```

In [9]:
video_id = "FPl-F2k_KtM"

In [10]:
# df = extract_transcript_df(video_id, extract_with_whisper=False)

In [11]:
# df.head()

In [12]:
# base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
# save_data_path = os.path.join(base_data_path, "yt", "raw_transcripts")
# DataProcessing.save_to_file(df, path=save_data_path, prefix=f'{video_id}', save_file_type='csv', include_version=False)

In [13]:
# change to "openai/whisper-large-v3" if more memory is available (openai/whisper-base, openai/whisper-small, openai/whisper-medium are other options)

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3", return_timestamps=True)

Device set to use cpu


In [14]:
df = extract_transcript_df(video_id, extract_with_whisper=True, whisper_pipe=pipe, audio_dir=r"..\data\yt\mp3_audio\sports")

c:\Users\adria\OneDrive\Área de Trabalho\UF Data Studio\predictions\.venv\Lib\site-packages\transformers\models\whisper\generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


In [16]:
df.head()

,Text,Start Time,Duration,Video ID
0,Hello and welcome to the Sharp 600 brought to you by Covers.com and presented by Bet365.,0.00,14.38,FPl-F2k_KtM
1,My name is Jason Logan. I will be your host here for the next 10 minutes as we take our first bites of Super Bowl 60 odds.,14.56,6.28,FPl-F2k_KtM
2,"And joining us for that dinner, for that snack time, is former odds maker, current professional bettor Todd Furman.",21.00,5.28,FPl-F2k_KtM
3,"Todd, happy bye week to you. The Seahawks, the Patriots get a little bit of a breather,",4.80,0.08,FPl-F2k_KtM
4,"a little bit of downtime this week, but I always found the bye was like my busiest time,",9.38,0.12,FPl-F2k_KtM


In [17]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
save_data_path = os.path.join(base_data_path, "yt", "whisper_transcripts", "sports")
DataProcessing.save_to_file(df, path=save_data_path, prefix=f'{video_id}', save_file_type='csv', include_version=False)

Saving CSV file to: c:\Users\adria\OneDrive\Área de Trabalho\UF Data Studio\predictions\prediction_acquition-youtube\../data\yt\whisper_transcripts\sports\FPl-F2k_KtM.csv
